# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kavanaaykavna/1stmlassignment/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


#Finding 1 — The Anatomy of Growing Content
The paper reports that growing pages tend to be longer and younger than declining pages. Growing pages averaged about 3.2K words and 184 days of age, while declining pages averaged about 2.3K words and 230 days of age. The paper also notes that this is an observational comparison.
My methodology question:
The paper defines trend direction using the change in impressions between the last 30 days and the previous 30 days, with Up meaning more than 10% growth and Down meaning more than 10% decline.
This means the finding describes an association between age, content length, and observed performance direction. It does not prove that making a page longer or younger will directly cause growth. Since the study is observational, I would treat this as a directional finding for prioritization rather than a causal claim.
#Finding 2 — The Content Performance Curve
The paper reports that content reaches its highest health score around 61–90 days, declines after 270 days, and that the 365+ recovery is concentrated among older pages that were refreshed.
My methodology question:
Age and freshness are separate variables in the study: age measures time since creation, while freshness measures time since the last update.
Because content age can confound model-performance comparisons, the observed relationship between age and performance should not automatically be interpreted as an effect of age itself.
Therefore, I would describe this as an observed lifecycle pattern. A stronger claim about the effect of refreshing content would require a controlled or time-aware comparison between refreshed and comparable unrefreshed pages.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

#Why I am changing the split
My Week 5 model used a random 80/20 split. For this audit, I use a client-grouped split so that pages from the same client are not mixed between training and test sets.
This is more conservative because the dataset contains multiple clients, and a random split can allow the model to learn client-specific patterns that may not generalize to a completely unseen client.
The research paper also works across multiple brands and notes that its ML analysis is exploratory and secondary to direct portfolio comparisons.

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Load dataset
df = pd.read_csv("content_refresh_anonymized.csv")

# Target
df["is_declining_label"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

print("Shape:", df.shape)
print(df["is_declining_label"].value_counts())
print("Number of clients:", df["client_id"].nunique())

Shape: (18698, 45)
is_declining_label
1    10072
0     8626
Name: count, dtype: int64
Number of clients: 32


In [3]:
# Features used in Week 5
features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

# Remove variables directly involved in constructing the target.
# trend_direction and trend_pct are also excluded.
leakage_features = [
    "impressions_last_30d",
    "impressions_prev_30d"
]

honest_features = [
    f for f in features
    if f not in leakage_features
]

print("Removed leakage-prone features:", leakage_features)
print("Number of honest features:", len(honest_features))

Removed leakage-prone features: ['impressions_last_30d', 'impressions_prev_30d']
Number of honest features: 27


### Leakage finding

The target `is_declining_label` is created from `trend_direction`.

The paper defines trend direction from the change in impressions between the last 30 days and the previous 30 days. Therefore, `impressions_last_30d` and `impressions_prev_30d` directly contain the information used to construct the target.

Using these variables as predictors allows the model to see information that is effectively part of the label construction. I therefore remove them from the audited model.

I also continue to exclude `trend_direction` and `trend_pct` because they directly encode the target or its calculation.

In [5]:
cleaned_df = df.dropna(subset=["client_id"])
X = cleaned_df[honest_features]
y = cleaned_df["is_declining_label"]
groups = cleaned_df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("Training clients:", cleaned_df.iloc[train_idx]["client_id"].nunique())
print("Test clients:", cleaned_df.iloc[test_idx]["client_id"].nunique())

print(
    "Overlapping clients:",
    len(
        set(cleaned_df.iloc[train_idx]["client_id"])
        & set(cleaned_df.iloc[test_idx]["client_id"])
    )
)

Training rows: 14834
Test rows: 3863
Training clients: 25
Test clients: 7
Overlapping clients: 0


In [6]:
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

honest_accuracy = accuracy_score(y_test, y_pred)
honest_precision = precision_score(y_test, y_pred)
honest_recall = recall_score(y_test, y_pred)
honest_f1 = f1_score(y_test, y_pred)

print("Honest Split Results")
print("--------------------")
print("Accuracy :", round(honest_accuracy, 4))
print("Precision:", round(honest_precision, 4))
print("Recall   :", round(honest_recall, 4))
print("F1       :", round(honest_f1, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Honest Split Results
--------------------
Accuracy : 0.5969
Precision: 0.6026
Recall   : 0.6048
F1       : 0.6037

Confusion Matrix:
[[1120  782]
 [ 775 1186]]

Classification Report:
              precision    recall  f1-score   support

           0       0.59      0.59      0.59      1902
           1       0.60      0.60      0.60      1961

    accuracy                           0.60      3863
   macro avg       0.60      0.60      0.60      3863
weighted avg       0.60      0.60      0.60      3863



## Before vs After

The Week 5 model used a random 80/20 split and included recent and previous 30-day impression features.

For this validation audit, I made two changes:

1. I removed `impressions_last_30d` and `impressions_prev_30d` because these variables are directly involved in constructing the decline label.
2. I used a client-grouped 80/20 split so that no client appears in both the training and test sets.

### Results

| Metric | Week 5 Random Split | Week 6 Honest Split |
|---|---:|---:|
| Accuracy | 0.8215 | 0.5969 |
| Precision | 0.8443 | 0.6026 |
| Recall | 0.8223 | 0.6048 |
| F1 Score | 0.8332 | 0.6037 |

The honest split produced lower performance than the original random split. This indicates that the original evaluation may have benefited from information overlap between training and test examples and from features closely related to the target construction.

The Week 6 model was evaluated on 3,863 test rows from 7 clients, with zero overlapping clients between training and test sets.

The honest-split results provide a more conservative estimate of model performance. Therefore, I would not use the Week 5 performance numbers as evidence of guaranteed performance on unseen clients.

The audited model should be treated as directional decision-support for prioritizing pages for SEO review rather than as a production-ready forecasting model.
### Honest Split Confusion Matrix

The confusion matrix was:

- True Negatives: 1,120
- False Positives: 782
- False Negatives: 775
- True Positives: 1,186

The model made both false-positive and false-negative errors at meaningful levels, reinforcing that model predictions should be reviewed alongside page-level SEO evidence.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## Leakage Audit

I checked the features used by my Week 5 model against the target definition.

### Direct leakage

`trend_direction` and `trend_pct` were excluded because they directly describe the target or its calculation.

More importantly, the target is based on the change between `impressions_last_30d` and `impressions_prev_30d`. Therefore, these two variables provide direct information about the label.

I removed:

- `impressions_last_30d`
- `impressions_prev_30d`

from the audited model.

### Potential information overlap

Some remaining performance variables, especially `impressions_90d`, contain recent performance information and may still overlap with the period used to define the target.

Because the available starter dataset is a snapshot rather than a full historical forecasting dataset, I cannot completely reconstruct a true future-prediction setup from the available fields.

Therefore, I do not claim that the audited model is a production-ready forecasting model.

### Conclusion

The main leakage issue identified was the use of variables directly involved in constructing the target. Removing them gives a more defensible validation setup.

The remaining limitation is that the dataset does not provide a complete historical feature timeline that would allow a fully time-forward prediction experiment.

In [7]:
# Show actual false positives and false negatives
test_results = df.iloc[test_idx].copy()

test_results["actual"] = y_test.values
test_results["predicted"] = y_pred

false_positives = test_results[
    (test_results["actual"] == 0) &
    (test_results["predicted"] == 1)
]

false_negatives = test_results[
    (test_results["actual"] == 1) &
    (test_results["predicted"] == 0)
]

print("False Positives:", len(false_positives))
print("False Negatives:", len(false_negatives))

print("\nExample False Positives:")
display(
    false_positives[
        [
            "content_id",
            "client_id",
            "word_count",
            "content_age_days",
            "days_since_last_update",
            "ctr",
            "avg_position",
            "actual",
            "predicted"
        ]
    ].head(10)
)

print("\nExample False Negatives:")
display(
    false_negatives[
        [
            "content_id",
            "client_id",
            "word_count",
            "content_age_days",
            "days_since_last_update",
            "ctr",
            "avg_position",
            "actual",
            "predicted"
        ]
    ].head(10)
)

False Positives: 782
False Negatives: 775

Example False Positives:


,content_id,client_id,word_count,content_age_days,days_since_last_update,ctr,avg_position,actual,predicted
13,content_a5a2fbc76336,client_8527a891e2,1342.0,238.0,103.0,0.00,39.8,0,1
26,content_72c5c2d73e5a,client_4e07408562,2686.0,300.0,13.0,0.12,30.0,0,1
36,content_bce275871a25,client_f369cb89fc,2510.0,187.0,20.0,1.35,5.4,0,1
56,content_dcebfd222b10,client_f369cb89fc,3158.0,145.0,20.0,0.00,4.6,0,1
64,content_685de0e3b7cb,client_f369cb89fc,2808.0,106.0,8.0,0.11,7.2,0,1
82,content_ec6fce716c78,client_4e07408562,2793.0,348.0,13.0,0.44,8.3,0,1
126,content_be5e23c0a35e,client_f369cb89fc,2492.0,126.0,20.0,0.00,21.0,0,1
179,content_552a9396d8dc,client_8527a891e2,3393.0,309.0,104.0,0.72,12.9,0,1
181,content_722d8cd002d3,client_f369cb89fc,2602.0,95.0,20.0,0.41,13.9,0,1
204,content_976d5deeab73,client_4e07408562,2873.0,445.0,25.0,0.10,19.0,0,1



Example False Negatives:


,content_id,client_id,word_count,content_age_days,days_since_last_update,ctr,avg_position,actual,predicted
23,content_2da6ae9d0882,client_e629fa6598,NaN,502.0,20.0,0.34,13.9,1,0
39,content_4595e8704e07,client_8527a891e2,3666.0,348.0,104.0,0.00,36.3,1,0
43,content_1938955b34c4,client_f369cb89fc,2394.0,138.0,20.0,0.00,2.9,1,0
47,content_40cb4af260c0,client_f369cb89fc,3086.0,126.0,20.0,0.00,25.5,1,0
49,content_f0717373e86e,client_8527a891e2,1585.0,174.0,8.0,0.00,10.1,1,0
54,content_ff8ea1364b59,client_e629fa6598,NaN,502.0,22.0,0.00,10.7,1,0
58,content_caff51984338,client_e629fa6598,NaN,494.0,20.0,0.00,9.1,1,0
60,content_b9104a222d01,client_f369cb89fc,2492.0,181.0,20.0,4.00,6.2,1,0
76,content_5607fec5d7db,client_f369cb89fc,2882.0,187.0,20.0,0.00,8.9,1,0
81,content_16788821b64a,client_e629fa6598,NaN,502.0,22.0,0.31,9.6,1,0


## Failure Interpretation

The false positives are pages that the model classified as declining but whose observed label was not declining. These examples show that pages with similar age, freshness, CTR, position, or traffic characteristics can still have different observed trend directions.

The false negatives are pages that were actually labelled as declining but were not identified by the model. These examples show that the available content and performance features do not fully explain every observed decline.

These errors are important because they show that the model should be used as a prioritization or decision-support tool rather than as an automatic replacement for SEO review.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Claim Rewrite

### Original claim

"My model predicts which webpages will decline in search performance."

### Revised claim

"My model identifies pages associated with an observed declining-impression label in the available dataset. Under a client-grouped validation split and after removing features directly involved in constructing the label, the model provides directional decision-support for prioritizing pages for further SEO review."

### Why I changed the claim

The original wording was too strong because the dataset is an observational snapshot and the target is constructed from observed impression changes.

The research paper also states that the study is observational and that correlations do not prove causation. Its ML analysis is described as exploratory and secondary to direct portfolio evidence. :contentReference[oaicite:6]{index=6}

Therefore, I use terms such as **observed**, **measured**, **directional**, and **decision-support** rather than claiming causal effects or guaranteed future prediction.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-Check

- [x] I reviewed two findings from the FlyRank research paper.
- [x] I identified methodology questions about label construction and validation.
- [x] I used a client-grouped split for a more conservative evaluation.
- [x] I checked the feature set for target leakage.
- [x] I removed variables directly involved in constructing the decline label.
- [x] I inspected false-positive and false-negative examples.
- [x] I avoided causal claims.
- [x] I rewrote the model claim using measured and decision-support language.
- [x] I documented limitations of the available snapshot dataset.

### Final takeaway

The validation audit showed that model performance depends not only on the algorithm but also on how the target and validation split are constructed. Removing target-derived features and testing on unseen client groups gives a more conservative and defensible evaluation. The resulting model should be treated as directional decision-support rather than a guaranteed forecasting system.